# RAPIDS single-cell clustering after SOPA segmentation

This notebook processes the `adata.h5ad` file produced by SOPA.

## Workflow

1. Load SOPA AnnData.
2. Standardize spatial coordinates.
3. Clean marker names.
4. Remove the composite channel from clustering input.
5. Use DAPI only for object-quality control.
6. Filter low-DAPI segmented objects using one allowed percentile: 10, 15, or 25.
7. Run the existing RAPIDS workflow:
   - arcsinh and log1p transformations
   - lineage-marker scaling
   - KNN graph construction
   - Leiden clustering over multiple resolutions
   - UMAP over multiple minimum distances
8. Save:
   - filtered and clustered AnnData
   - DAPI QC tables and figures
   - clustering tables
   - session-summary tables

## Technical note

DAPI is not used as a clustering marker.  
DAPI is used to remove likely non-nucleated (red blood cells) or autofluorescent segmented objects before clustering.

In [ ]:
# ==============================================================================
# PARAMETERS
# ==============================================================================
# These are placeholder values.
# In the automated pipeline, Nextflow + Papermill overrides them.
#
# Required external parameters:
#   SAMPLE_INDEX:
#       Order of the sample in the SOPA/analysis manifest.
#
#   SAMPLE_ID:
#       Biological/sample identifier from the master samplesheet.
#
#   SAMPLE_DIR_NAME:
#       Safe output folder name generated by the workflow.
#       Example: 001_Cirrhotic_MASLD
#
#   INPUT_PATH:
#       Path to SOPA output:
#       outputs/02_sopa/<sample>.explorer/adata.h5ad
#
#   OUTPUT_DIR:
#       Sample-specific output folder:
#       outputs/03_RAPIDS_clustering/<sample_dir_name>
#
#   DAPI_QC_PERCENTILE:
#       Only allowed values: 10, 15, 25.
# ==============================================================================

SAMPLE_INDEX = 1
SAMPLE_ID = "sample"
SAMPLE_DIR_NAME = "001_sample"

INPUT_PATH = "adata.h5ad"
OUTPUT_DIR = "rapids_output"

# Only allowed values: 10, 15, 25
DAPI_QC_PERCENTILE = 15

# Existing RAPIDS workflow parameters
COFACTOR = 5.0
N_NEIGHBORS = 15
NORMALIZATIONS = ["arcsinh", "log1p"]
RESOLUTIONS = [0.5, 1.0, 1.5, 2.0]
MIN_DISTANCES = [0.1, 0.3, 0.5]

In [ ]:
# ==============================================================================
# SECTION 1: IMPORTS AND OUTPUT DIRECTORIES
# ==============================================================================

import re
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import rapids_singlecell as rsc

from great_tables import GT, md as gt_md


# ------------------------------------------------------------------------------
# Validate DAPI QC percentile
# ------------------------------------------------------------------------------

ALLOWED_DAPI_QC_PERCENTILES = [10, 15, 25]

if DAPI_QC_PERCENTILE not in ALLOWED_DAPI_QC_PERCENTILES:
    raise ValueError(
        f"DAPI_QC_PERCENTILE must be one of {ALLOWED_DAPI_QC_PERCENTILES}. "
        f"Got: {DAPI_QC_PERCENTILE}"
    )


# ------------------------------------------------------------------------------
# Create output directories
# ------------------------------------------------------------------------------

OUTPUT_DIR = Path(OUTPUT_DIR)

QC_DIR = OUTPUT_DIR / "dapi_qc"
CLUSTERING_DIR = OUTPUT_DIR / "clustering"
REPORT_DIR = OUTPUT_DIR / "report"

QC_DIR.mkdir(parents=True, exist_ok=True)
CLUSTERING_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


print("RAPIDS clustering notebook initialized")
print(f"SAMPLE_INDEX:        {SAMPLE_INDEX}")
print(f"SAMPLE_ID:           {SAMPLE_ID}")
print(f"SAMPLE_DIR_NAME:     {SAMPLE_DIR_NAME}")
print(f"INPUT_PATH:          {INPUT_PATH}")
print(f"OUTPUT_DIR:          {OUTPUT_DIR}")
print(f"DAPI_QC_PERCENTILE:  {DAPI_QC_PERCENTILE}")

In [ ]:
# ==============================================================================
# SECTION 2: LOAD SOPA ANNDATA
# ==============================================================================

adata = ad.read_h5ad(INPUT_PATH)

print(f"Loaded AnnData: {adata.n_obs:,} objects × {adata.n_vars} markers")
print(adata)

print("\nOriginal markers:")
print(list(adata.var_names))


# ------------------------------------------------------------------------------
# Standardize spatial coordinates
# ------------------------------------------------------------------------------

if "spatial" in adata.obsm:
    adata.obs["X_centroid"] = adata.obsm["spatial"][:, 0]
    adata.obs["Y_centroid"] = adata.obsm["spatial"][:, 1]

elif {"X_centroid", "Y_centroid"}.issubset(adata.obs.columns):
    adata.obsm["spatial"] = adata.obs[["X_centroid", "Y_centroid"]].values

else:
    print("WARNING: no spatial coordinates found.")


# ------------------------------------------------------------------------------
# Store sample identifiers
# ------------------------------------------------------------------------------

adata.obs["sample_id"] = SAMPLE_ID
adata.obs["sample_index"] = SAMPLE_INDEX
adata.obs["sample_dir_name"] = SAMPLE_DIR_NAME

if "slide" not in adata.obs.columns:
    adata.obs["slide"] = SAMPLE_ID


# Snapshot used later for the session-summary table
initial_obs = list(adata.obs.columns)
initial_obsm = list(adata.obsm.keys())
initial_layers = list(adata.layers.keys())

In [ ]:
# ==============================================================================
# SECTION 3: MARKER CLEANUP, COMPOSITE REMOVAL, AND LINEAGE SELECTION
# ==============================================================================
# This keeps the original logic of the previous RAPIDS notebook:
#   - clean marker names
#   - remove composite channel entirely
#   - keep DAPI in AnnData but exclude it from lineage/clustering markers
# ==============================================================================

def simplify_marker_name(name: str) -> str:
    """
    Simplify marker names while preserving biological readability.

    Examples:
        "HLA-DR"       -> "HLA_DR"
        "Collagen 3"   -> "Collagen_3"
        "PD-1 - Cy5"   -> "PD_1"
    """

    s = str(name).strip()

    s = re.sub(r"\s*\(.*?\)\s*", "", s)
    s = re.sub(r"\s*\[.*?\]\s*", "", s)

    parts = re.split(r"\s-\s", s, maxsplit=1)
    s = re.sub(r"[^a-zA-Z0-9]+", "_", parts[0].strip())

    return re.sub(r"_+", "_", s).strip("_")


def make_unique(names):
    """
    Make marker names unique if duplicated.
    """

    seen = {}
    out = []

    for n in names:
        if n not in seen:
            seen[n] = 0
            out.append(n)
        else:
            seen[n] += 1
            out.append(f"{n}_{seen[n]}")

    return out


def is_likely_technical_channel(clean_name: str) -> bool:
    """
    Identify technical channels excluded from phenotyping/clustering.

    DAPI is kept in the object for QC/visualization but excluded from clustering.
    Composite channels should already be removed before this step.
    """

    s = str(clean_name).lower()

    technical_terms = {
        "dapi",
        "composite_mip",
    }

    return any(term in s for term in technical_terms)


# ------------------------------------------------------------------------------
# Clean marker names
# ------------------------------------------------------------------------------

marker_table = pd.DataFrame(index=adata.var_names.copy())
marker_table["marker_original"] = marker_table.index.astype(str)
marker_table["marker_clean"] = make_unique(
    marker_table["marker_original"].map(simplify_marker_name)
)

adata.var = marker_table.copy()
adata.var_names = adata.var["marker_clean"].tolist()

print("Cleaned marker names:")
print(list(adata.var_names))


# ------------------------------------------------------------------------------
# Remove composite channel entirely
# ------------------------------------------------------------------------------

composite_mask = adata.var_names.str.lower().str.contains("composite")

if composite_mask.any():
    composite_names = adata.var_names[composite_mask].tolist()
    adata = adata[:, ~composite_mask].copy()
    print(f"\nComposite channel(s) removed from AnnData: {composite_names}")
else:
    print("\nNo composite channel detected.")


# ------------------------------------------------------------------------------
# Mark technical and lineage channels
# ------------------------------------------------------------------------------

adata.var["exclude_from_phenotyping"] = adata.var_names.map(
    is_likely_technical_channel
)

adata.uns["all_markers"] = adata.var_names.tolist()
adata.uns["technical_channels"] = (
    adata.var_names[adata.var["exclude_from_phenotyping"]].tolist()
)
adata.uns["lineage_markers"] = (
    adata.var_names[~adata.var["exclude_from_phenotyping"]].tolist()
)

print("\nTechnical channels kept in AnnData but excluded from clustering:")
print(adata.uns["technical_channels"])

print("\nLineage markers used for clustering:")
print(adata.uns["lineage_markers"])

In [ ]:
# ==============================================================================
# SECTION 4: DAPI-BASED OBJECT QUALITY CONTROL
# ==============================================================================
# Goal:
#   Remove likely non-nucleated segmentation objects before clustering.
#
# Rationale:
#   Some Cellpose-SAM objects may correspond to red blood cells or
#   autofluorescent regions. These objects can be bright in many marker channels
#   but weak in DAPI.
#
# Important:
#   DAPI is used only for QC.
#   DAPI is not used as a clustering marker.
#
# Allowed thresholds:
#   DAPI_QC_PERCENTILE must be one of: 10, 15, 25
# ==============================================================================

def get_marker_vector_by_name_or_first_channel(adata_obj, marker_name="DAPI"):
    """
    Extract DAPI values.

    Preferred:
        use marker called DAPI

    Fallback:
        use first channel, because in this workflow DAPI is expected at channel 0.
    """

    if marker_name in adata_obj.var_names:
        marker_idx = list(adata_obj.var_names).index(marker_name)
        resolved_marker = marker_name
    else:
        marker_idx = 0
        resolved_marker = str(adata_obj.var_names[0])
        print(
            f"WARNING: marker '{marker_name}' not found exactly. "
            f"Using first channel instead: '{resolved_marker}'"
        )

    X = adata_obj.X
    values = X[:, marker_idx]

    if sp.issparse(values):
        values = values.toarray()

    values = np.asarray(values).ravel().astype(float)

    return values, resolved_marker, marker_idx


# ------------------------------------------------------------------------------
# Extract DAPI and apply the selected percentile threshold
# ------------------------------------------------------------------------------

dapi_values, dapi_resolved_marker, dapi_marker_index = (
    get_marker_vector_by_name_or_first_channel(adata, marker_name="DAPI")
)

dapi_threshold = float(np.nanpercentile(dapi_values, DAPI_QC_PERCENTILE))
qc_keep = dapi_values >= dapi_threshold

n_total = int(adata.n_obs)
n_kept = int(qc_keep.sum())
n_removed = int((~qc_keep).sum())


# ------------------------------------------------------------------------------
# Store QC variables in AnnData
# ------------------------------------------------------------------------------
# These variables are saved in the final AnnData.
# No kept/dropped cell-ID CSVs are written.
# ------------------------------------------------------------------------------

adata.obs["qc_dapi_mean"] = dapi_values
adata.obs["qc_dapi_percentile"] = DAPI_QC_PERCENTILE
adata.obs["qc_dapi_threshold"] = dapi_threshold
adata.obs["qc_keep_for_analysis"] = qc_keep
adata.obs["qc_removed_low_dapi"] = ~qc_keep

adata.uns["dapi_qc"] = {
    "qc_applied": True,
    "allowed_dapi_exclusion_percentiles": [10, 15, 25],
    "applied_dapi_exclusion_percentile": int(DAPI_QC_PERCENTILE),
    "dapi_requested_marker": "DAPI",
    "dapi_resolved_marker": dapi_resolved_marker,
    "dapi_marker_index": int(dapi_marker_index),
    "dapi_threshold": float(dapi_threshold),
    "total_segmented_objects_before_qc": n_total,
    "kept_objects_after_qc": n_kept,
    "dropped_objects_low_dapi": n_removed,
    "dropped_fraction_low_dapi": float(n_removed / n_total),
    "dropped_percent_low_dapi": float(100 * n_removed / n_total),
    "final_adata_is_dapi_qc_filtered": False,
}


# ------------------------------------------------------------------------------
# Compact report table
# ------------------------------------------------------------------------------
# This table is only displayed and later saved as PNG in SECTION 5.
# It is not saved as CSV.
# ------------------------------------------------------------------------------

qc_summary = pd.DataFrame([
    {
        "sample_id": SAMPLE_ID,
        "dapi_marker": dapi_resolved_marker,
        "allowed_exclusion_percentiles": "10, 15, 25",
        "applied_exclusion_percentile": DAPI_QC_PERCENTILE,
        "dapi_threshold": dapi_threshold,
        "total_objects": n_total,
        "kept_objects": n_kept,
        "dropped_objects": n_removed,
        "dropped_percent": 100 * n_removed / n_total,
    }
])

print("DAPI QC summary:")
display(qc_summary)

print()
print("Allowed DAPI exclusion percentiles: 10, 15, 25")
print(f"Applied DAPI exclusion percentile:  {DAPI_QC_PERCENTILE}")
print(f"DAPI threshold:                     {dapi_threshold:.4f}")
print(f"Total segmented objects:            {n_total:,}")
print(f"Kept objects:                       {n_kept:,}")
print(f"Dropped objects:                    {n_removed:,}")
print(f"Dropped percent:                    {100 * n_removed / n_total:.2f}%")

In [ ]:
# ==============================================================================
# SECTION 5: DAPI QC FIGURES AND TABLE IMAGE
# ==============================================================================

def save_table_png(df, path, title=None, dpi=300):
    """
    Save a pandas DataFrame as a high-resolution PNG image.
    """

    df_to_show = df.copy()

    for col in df_to_show.columns:
        if pd.api.types.is_float_dtype(df_to_show[col]):
            df_to_show[col] = df_to_show[col].round(4)

    n_rows, n_cols = df_to_show.shape

    fig_width = max(10, n_cols * 1.5)
    fig_height = max(2.8, n_rows * 0.6 + 1.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    if title is not None:
        ax.set_title(title, fontsize=12, pad=12)

    table = ax.table(
        cellText=df_to_show.astype(str).values,
        colLabels=df_to_show.columns,
        loc="center",
        cellLoc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.4)

    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


# ------------------------------------------------------------------------------
# DAPI histogram
# ------------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(dapi_values, bins=120, alpha=0.85)
ax.axvline(
    dapi_threshold,
    linestyle="--",
    linewidth=2,
    label=f"{DAPI_QC_PERCENTILE}th percentile threshold",
)

ax.set_title(f"{SAMPLE_ID} - DAPI QC histogram")
ax.set_xlabel("DAPI mean intensity per segmented object")
ax.set_ylabel("Segmented object count")
ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(QC_DIR / "dapi_histogram.png", dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------------------------
# Spatial DAPI QC map: kept and dropped objects
# ------------------------------------------------------------------------------

if "spatial" in adata.obsm:
    coords = adata.obsm["spatial"]

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.scatter(
        coords[qc_keep, 0],
        coords[qc_keep, 1],
        s=0.2,
        alpha=0.25,
        label="kept after DAPI QC",
    )

    ax.scatter(
        coords[~qc_keep, 0],
        coords[~qc_keep, 1],
        s=0.2,
        alpha=0.60,
        label="dropped low-DAPI",
    )

    ax.set_title(f"{SAMPLE_ID} - DAPI QC spatial map")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.invert_yaxis()
    ax.legend(markerscale=10, frameon=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(QC_DIR / "dapi_spatial_qc.png", dpi=300, bbox_inches="tight")
    plt.show()

else:
    print("No spatial coordinates available; DAPI QC spatial plot skipped.")


    # ------------------------------------------------------------------------------
# QC summary table as PNG
# ------------------------------------------------------------------------------

save_table_png(
    qc_summary,
    QC_DIR / "dapi_qc_summary.png",
    title=f"{SAMPLE_ID} - DAPI QC summary",
    dpi=300,
)

print("DAPI QC outputs saved to:")
print(QC_DIR)

In [ ]:
# ==============================================================================
# SECTION 6: APPLY DAPI QC FILTER
# ==============================================================================
# From this point onward, adata contains only QC-passing cells/objects.
# This keeps the rest of the RAPIDS code close to the original notebook.
# ==============================================================================

adata = adata[adata.obs["qc_keep_for_analysis"].values].copy()

adata.uns["dapi_qc"]["final_adata_is_dapi_qc_filtered"] = True

print("AnnData after DAPI QC filtering:")
print(adata)

In [ ]:
# ==============================================================================
# SECTION 7: RAPIDS GPU PROCESSING AFTER DAPI QC FILTERING
# ==============================================================================
# This section keeps the same logic as the original RAPIDS notebook:
#   - protect raw matrix
#   - compute arcsinh and log1p transformations
#   - store full transformed layers
#   - slice lineage markers
#   - push sliced object to GPU
#   - scale lineage representation
#   - build neighbors on lineage-only representation
#   - run Leiden for all selected resolutions
#   - run UMAP for all selected minimum-distance values
#
# Important:
#   adata has already been filtered by DAPI QC.
# ==============================================================================

pre_rapids_obs = list(adata.obs.columns)
pre_rapids_obsm = list(adata.obsm.keys())
pre_rapids_layers = list(adata.layers.keys())

raw_X = adata.X.copy()


def gpu_slice(adata_obj, var_keys):
    """
    Slice markers and immediately push result to GPU.

    AnnData slicing/copying can lose the GPU-backed array context,
    so the sliced object is explicitly moved to GPU before RAPIDS operations.
    """

    subset = adata_obj[:, var_keys].copy()
    rsc.get.anndata_to_GPU(subset)
    return subset


for norm in NORMALIZATIONS:
    print(f"\n========== Processing normalization: {norm.upper()} ==========")

    # 1.
    adata.X = raw_X.copy()

    # 2. Normalize full object
    if norm == "arcsinh":
        adata.X = np.arcsinh(adata.X / COFACTOR)

    elif norm == "log1p":
        sc.pp.log1p(adata)

    else:
        raise ValueError(f"Unknown normalization: {norm}")

    adata.layers[f"{norm}_full"] = adata.X.copy()

    # 3. Slice lineage markers and push to GPU
    adata_lineage = gpu_slice(adata, adata.uns["lineage_markers"])

    # 4. Scale lineage representation on GPU
    rsc.pp.scale(adata_lineage, max_value=10)

    # 5. Store scaled lineage representation in main AnnData on CPU
    rep_key = f"X_{norm}_scaled_lineage"
    adata.obsm[rep_key] = adata_lineage.X.get()

    # 6. Push main object to GPU for graph, UMAP, and Leiden
    print("  -> Pushing main object to GPU for structural math...")
    rsc.get.anndata_to_GPU(adata)

    # 7. Build KNN graph using lineage-only representation
    print("  -> Building KNN graph on lineage markers...")
    rsc.pp.neighbors(
        adata,
        n_neighbors=N_NEIGHBORS,
        use_rep=rep_key,
    )

    # 8. Leiden clustering
    print("  -> Running Leiden clustering...")
    for res in RESOLUTIONS:
        res_str = str(res).replace(".", "_")
        rsc.tl.leiden(
            adata,
            resolution=res,
            key_added=f"leiden_{norm}_res_{res_str}",
        )

    # 9. UMAP generation
    print("  -> Running UMAP generation...")
    for dist in MIN_DISTANCES:
        dist_str = str(dist).replace(".", "_")
        rsc.tl.umap(adata, min_dist=dist)
        adata.obsm[f"X_umap_{norm}_md_{dist_str}"] = adata.obsm["X_umap"].copy()

    # 10. Return main object to CPU before next loop or saving
    print("  -> Returning main object to CPU memory...")
    rsc.get.anndata_to_CPU(adata)


# Clean up default RAPIDS UMAP key
if "X_umap" in adata.obsm.keys():
    del adata.obsm["X_umap"]

## Technical note on RAPIDS and AnnData slicing

The notebook explicitly moves sliced objects back to GPU before RAPIDS operations.

Pattern used:

```python
adata_lineage = adata[:, lineage_markers].copy()
rsc.get.anndata_to_GPU(adata_lineage)
rsc.pp.scale(adata_lineage)

In [ ]:
# ==============================================================================
# SECTION 8: SAVE FINAL FILTERED + CLUSTERED ANNDATA
# ==============================================================================

adata.uns["rapids_clustering"] = {
    "sample_index": int(SAMPLE_INDEX),
    "sample_id": SAMPLE_ID,
    "sample_dir_name": SAMPLE_DIR_NAME,
    "normalizations": NORMALIZATIONS,
    "resolutions": RESOLUTIONS,
    "min_distances": MIN_DISTANCES,
    "n_neighbors": int(N_NEIGHBORS),
    "cofactor": float(COFACTOR),
    "lineage_markers": adata.uns["lineage_markers"],
    "technical_channels": adata.uns["technical_channels"],
    "dapi_qc_applied": True,
    "dapi_qc_percentile": int(DAPI_QC_PERCENTILE),
    "dapi_qc_threshold": float(dapi_threshold),
}

FINAL_ADATA_PATH = CLUSTERING_DIR / "rapids_clustered.h5ad"

adata.write_h5ad(FINAL_ADATA_PATH)

print("Pipeline complete.")
print(f"Filtered + clustered AnnData saved to:\n{FINAL_ADATA_PATH}")
print(adata)

In [ ]:
# ==============================================================================
# SECTION 10: SESSION SUMMARY TABLE
# ==============================================================================

new_obs = [k for k in adata.obs.columns if k not in initial_obs]
new_obsm = [k for k in adata.obsm.keys() if k not in initial_obsm]
new_layers = [k for k in adata.layers.keys() if k not in initial_layers]

rows = []

for k in new_layers:
    rows.append({
        "Slot": ".layers",
        "Variable Name": k,
        "Format / Shape": f"Matrix {adata.layers[k].shape}",
    })

for k in new_obsm:
    rows.append({
        "Slot": ".obsm",
        "Variable Name": k,
        "Format / Shape": f"Array {adata.obsm[k].shape}",
    })

for k in new_obs:
    rows.append({
        "Slot": ".obs",
        "Variable Name": k,
        "Format / Shape": f"Series ({adata.n_obs},)",
    })

session_summary = pd.DataFrame(rows)



save_table_png(
    session_summary,
    REPORT_DIR / "session_summary.png",
    title=f"{SAMPLE_ID} - variables generated in RAPIDS notebook",
    dpi=300,
)

table = (
    GT(session_summary)
    .tab_header(
        title=gt_md("**New Variables Generated in this Session**"),
        subtitle="Tracking calculations appended to the filtered AnnData object",
    )
    .cols_label(
        **{
            "Slot": gt_md("**AnnData Slot**"),
            "Variable Name": gt_md("**Variable Name**"),
            "Format / Shape": gt_md("**Shape**"),
        }
    )
    .tab_options(table_background_color="white", heading_align="left")
)

table